In [1]:

import sys
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import DataLoader
from torch.amp import autocast, GradScaler
from skimage.metrics import structural_similarity, peak_signal_noise_ratio
from tqdm import tqdm

# --------------------------------------------------
# 1. Project configuration
# --------------------------------------------------

ROOT = Path(r"D:\PS07_CT2MRI")

sys.path.insert(0, str(ROOT / "src"))

from multi_patient_loader import (
    MultiPatientDataset,
    PatientBatchSampler
)

from resunet import ResidualUNet


DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

BATCH_SIZE = 1
EPOCHS = 10
LEARNING_RATE = 1e-4
SEED = 42

MODELS_DIR = ROOT / "models"
RESULTS_DIR = ROOT / "results"

MODELS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)


# --------------------------------------------------
# 2. Reproducibility
# --------------------------------------------------

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# --------------------------------------------------
# 3. Loss function
# --------------------------------------------------

def masked_l1_loss(prediction, target, mask):

    mask = (mask > 0).float()

    masked_voxels = mask.sum()

    # Empty-mask sample: use ordinary L1
    if masked_voxels.item() == 0:
        return torch.abs(prediction - target).mean()

    # Non-empty sample: masked L1
    error = torch.abs(prediction - target) * mask

    return error.sum() / masked_voxels


# --------------------------------------------------
# 4. Validation metrics
# --------------------------------------------------

def calculate_metrics(prediction, target, mask):

    prediction = np.clip(prediction, 0, 1)
    target = np.clip(target, 0, 1)

    mask = mask > 0

    full_mae = np.mean(
        np.abs(prediction - target)
    )

    if np.any(mask):
        masked_mae = np.mean(
            np.abs(prediction[mask] - target[mask])
        )
    else:
        masked_mae = np.nan

    ssim = structural_similarity(
        target,
        prediction,
        data_range=1.0
    )

    psnr = peak_signal_noise_ratio(
        target,
        prediction,
        data_range=1.0
    )

    return masked_mae, full_mae, ssim, psnr


# --------------------------------------------------
# 5. Validation
# --------------------------------------------------

@torch.no_grad()
def validate(model, val_loader):

    model.eval()

    patient_results = {}

    for ct, mr, mask in tqdm(
        val_loader,
        desc="Validation",
        leave=False
    ):

        ct = ct.to(DEVICE)
        mr = mr.to(DEVICE)
        mask = mask.to(DEVICE)

        with autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda")
        ):

            prediction = model(ct)

        prediction = prediction.float()

        # Patient batches are grouped together
        patient_index = (
            val_loader.batch_sampler.dataset.offsets
        )

        # The current batch belongs to one patient
        # Identify it using the batch's sample index
        # through the validation sampler's current order.
        #
        # Patient identity is assigned below using
        # the sequential patient batch traversal.
        raise RuntimeError(
            "Patient-aware validation loop needs the "
            "patient batch iterator defined in Step 2 below."
        )


# --------------------------------------------------
# 6. Training setup
# --------------------------------------------------

print("Device:", DEVICE)

train_dataset = MultiPatientDataset(
    ROOT / "train_patients.csv"
)

val_dataset = MultiPatientDataset(
    ROOT / "val_patients.csv"
)

train_sampler = PatientBatchSampler(
    train_dataset,
    BATCH_SIZE,
    shuffle_patients=True
)

train_loader = DataLoader(
    train_dataset,
    batch_sampler=train_sampler,
    num_workers=0
)

model = ResidualUNet(
    in_channels=3,
    out_channels=1,
    base_channels=16
).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE
)

scaler = GradScaler(
    "cuda",
    enabled=(DEVICE.type == "cuda")
)

criterion = masked_l1_loss

print("\nModel:", model.__class__.__name__)
print("Parameters:", sum(p.numel() for p in model.parameters()))
print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Batch size:", BATCH_SIZE)


# --------------------------------------------------
# 7. Training loop
# --------------------------------------------------

history = []

for epoch in range(EPOCHS):

    model.train()

    running_loss = 0.0

    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}"
    )

    for ct, mr, mask in progress:

        ct = ct.to(DEVICE)
        mr = mr.to(DEVICE)
        mask = mask.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        with autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda")
        ):

            prediction = model(ct)
            loss = criterion(prediction, mr, mask)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()

        progress.set_postfix(
            loss=f"{loss.item():.5f}"
        )

    train_loss = running_loss / len(train_loader)

    print(
        f"\nEpoch {epoch + 1}: "
        f"Training Loss = {train_loss:.6f}"
    )

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss
    })

    # Save latest checkpoint after every epoch
    torch.save({
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "train_loss": train_loss
    }, MODELS_DIR / "latest_checkpoint.pth")

    pd.DataFrame(history).to_csv(
        RESULTS_DIR / "training_history.csv",
        index=False
    )

print("\nTraining loop completed.")


Device: cuda

Preparing patient slice indexes...
P1046: 179 masked + 4 background = 183 selected slices
P1068: 182 masked + 3 background = 185 selected slices
P1127: 176 masked + 5 background = 181 selected slices
P1316: 182 masked + 3 background = 185 selected slices
P1375: 178 masked + 4 background = 182 selected slices
P1389: 181 masked + 3 background = 184 selected slices
P1815: 176 masked + 5 background = 181 selected slices
P1965: 179 masked + 4 background = 183 selected slices
P2017: 181 masked + 3 background = 184 selected slices
P2052: 181 masked + 3 background = 184 selected slices
P2161: 180 masked + 4 background = 184 selected slices
P2341: 180 masked + 4 background = 184 selected slices
P2349: 176 masked + 5 background = 181 selected slices
P2374: 181 masked + 3 background = 184 selected slices
P2420: 180 masked + 4 background = 184 selected slices
P2622: 181 masked + 3 background = 184 selected slices
P2710: 181 masked + 3 background = 184 selected slices
P2728: 177 maske

Epoch 1/10: 100%|██████████| 20965/20965 [14:35<00:00, 23.94it/s, loss=0.06523]



Epoch 1: Training Loss = 0.076731


Epoch 2/10: 100%|██████████| 20965/20965 [13:47<00:00, 25.32it/s, loss=0.03941]



Epoch 2: Training Loss = 0.063733


Epoch 3/10: 100%|██████████| 20965/20965 [15:22<00:00, 22.74it/s, loss=0.07757]  


... (11 outputs left out)

Epoch 9/10: 100%|██████████| 20965/20965 [14:29<00:00, 24.12it/s, loss=0.00186] 



Epoch 9: Training Loss = 0.049936


Epoch 10/10: 100%|██████████| 20965/20965 [12:46<00:00, 27.35it/s, loss=0.06074]



Epoch 10: Training Loss = 0.049165

Training loop completed.
